# Predictive Cardiology: Heart Failure Mortality Prediction Model

**Aim:** To bridge clinical pathology and data science by performing exploratory data analysis and building a Machine Learning pipeline capable of predicting patient mortality risk based on standard cardiovascular vitals.

**Clinical Context:**

In cardiology, assessing heart failure risk relies on biomarkers like ejection fraction, serum creatinine, and blood pressure. In this project, we treat these clinical features as our input data. We will train two distinct algorithms, a Logistic Regression model (acting as a Clinical Risk Score) and a Random Forest Classifier (acting as a Multidisciplinary Medical Board), to analyze these vitals and predict the definitive outcome (Survival vs. Death Event).

### **Explanation of Step 1**
Before we can operate on the data, the environment needs to be prepared. The following were imported:
* **Pandas & NumPy:** To handle the electronic "patient files" and manipulate the data structures.
* **Scikit-Learn:** The core machine learning library that contains the predictive algorithms and data scaling tools.

In [ ]:
# Import necesssary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Necessary libraries prepped and ready.")

Necessary libraries prepped and ready.


### **Explanation of Step 2**

Here, we imported the clinical dataset. This dataset tracks 13 specific clinical features (including age, anemia status, diabetes, ejection fraction, and high blood pressure) for patients diagnosed with cardiovascular disease. Loading this into a Pandas DataFrame is the equivalent of opening the master patient chart.

In [ ]:
# Load the patient records into a DataFrame
df = pd.read_csv('Heart_failure_clinical_records_dataset.csv')

print("Patient files successfully loaded!")

Patient files successfully loaded!


### **Explanation of Step 3 (Exploratory Data Analysis)**

Before training an algorithm, the dataset is inspected.

We used `.info()` and `.describe()` to:
1. Verify the total number of patients.
2. Check for missing values (NaNs). If data is missing, we must impute it to prevent the algorithm from crashing.
3. Review the statistical spread of the vitals (e.g., identifying the average ejection fraction across our patient population).


In [ ]:
# 1. Look at the first 5 patient records to see the columns
print("--- First 5 Patient Records ---")
display(df.head())

# 2. See the total number of patients, features, and check for missing data
print("\n--- Dataset Vitals ---")
df.info()

# 3. Look at the statistical spread (average age, max ejection fraction, etc.)
print("\n--- Statistical Summary ---")
display(df.describe())

--- First 5 Patient Records ---


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1



--- Dataset Vitals ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
count,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.00000,299.000000,299.000000,299.00000,299.000000,299.00000
mean,60.837237,0.431438,581.839465,0.418060,38.083612,0.351171,263358.029264,1.39388,136.625418,0.648829,0.32107,130.260870,0.32107
std,11.900919,0.496107,970.287881,0.494067,11.834841,0.478136,97804.236869,1.03451,4.412477,0.478136,0.46767,77.614208,0.46767
min,40.000000,0.000000,23.000000,0.000000,14.000000,0.000000,25100.000000,0.50000,113.000000,0.000000,0.00000,4.000000,0.00000
25%,51.000000,0.000000,116.500000,0.000000,30.000000,0.000000,212500.000000,0.90000,134.000000,0.000000,0.00000,73.000000,0.00000
50%,60.000000,0.000000,250.000000,0.000000,38.000000,0.000000,262000.000000,1.10000,137.000000,1.000000,0.00000,115.000000,0.00000
75%,70.000000,1.000000,582.000000,1.000000,45.000000,1.000000,303500.000000,1.40000,140.000000,1.000000,1.00000,203.000000,1.00000
max,95.000000,1.000000,7861.000000,1.000000,80.000000,1.000000,850000.000000,9.40000,148.000000,1.000000,1.00000,285.000000,1.00000


### **Explanation of Step 4**

Machine learning models require perfectly balanced numerical inputs.
* **The Split:** We separated our data into an 80% Training Set (historical case files for the algorithm to study) and a 20% Testing Set (unseen patients to verify diagnostic accuracy).
* **Standardization:** A normal platelet count might be 250,000, while serum creatinine is around 1.0. If we feed raw numbers to an algorithm, it might mistakenly prioritize platelets simply because the number is larger. `StandardScaler` was used to mathematically balance all vitals so they are evaluated fairly.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Define the Target (The Prognosis) and the Features (The Vitals)
# (Target column is the 'DEATH_EVENT')
X = df.drop('DEATH_EVENT', axis=1) # Inputs: All columns EXCEPT the death event
y = df['DEATH_EVENT']              # Output: ONLY the death event column

# Check how many patients survived (0) vs passed away (1)
print("--- Patient Outcome Distribution ---")
print(y.value_counts())

# 2. Split the Data (80% for studying, 20% for the final exam)
# Using stratify=y to ensure the 80/20 split has the exact same ratio of survivors to non-survivors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Scale the Vitals (Standardization)
scaler = StandardScaler()

# We 'fit' the scaler only on the training data to prevent the model from peeking at the test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Only transform the test data

print("\n--- Data Prep Complete ---")
print(f"Training on {X_train.shape[0]} patients.")
print(f"Testing on {X_test.shape[0]} unseen patients.")
print("Vitals successfully scaled. The field is ready for the algorithm!")

--- Patient Outcome Distribution ---
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

--- Data Prep Complete ---
Training on 239 patients.
Testing on 60 unseen patients.
Vitals successfully scaled. The field is ready for the algorithm!


### **Explanation of Step 5**

**Algorithm Initialization & Training**

We deployed two distinct machine learning models to compare their diagnostic accuracy:
1. **Logistic Regression (The Clinical Risk Score):** A mathematical equation that assigns a specific weight to each vital sign, similar to the CHADS₂ score for stroke risk. Highly interpretable.
2. **Random Forest Classifier (The Multidisciplinary Team):** An ensemble model that builds 100 different decision trees. It acts like a board of 100 doctors independently reviewing the chart and going with the majority vote. Highly accurate for complex data.


**Final Diagnosis & Clinical Evaluation**

The models have learned the patterns. Now, we test them on the 20% of unseen patients. In clinical data science, raw accuracy is not enough. We must evaluate:
* **Precision:** Minimizing false positives (We don't want to needlessly panic a stable patient).
* **Recall (Sensitivity):** Minimizing false negatives (We absolutely cannot miss a critical patient who is at high risk of a mortality event).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Initialize the Algorithms
log_reg = LogisticRegression(random_state=42)
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the Models (Study the 80% training data)
print("Training Logistic Regression (Clinical Risk Score)...")
log_reg.fit(X_train_scaled, y_train)

print("Training Random Forest (Multidisciplinary Team)...")
random_forest.fit(X_train_scaled, y_train)
print("Training complete! \n")

# 3. Make Diagnoses on the Unseen Test Patients (The 20% test data)
log_pred = log_reg.predict(X_test_scaled)
rf_pred = random_forest.predict(X_test_scaled)

# 4. Evaluate the Results (How many patients did they diagnose correctly?)
print("--- Diagnostic Accuracy ---")
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, log_pred) * 100:.2f}%")
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_pred) * 100:.2f}%\n")

# 5. Look deeper into the Random Forest's performance
print("--- Detailed Clinical Report (Random Forest) ---")
# 0 = Survived, 1 = Passed Away
print(classification_report(y_test, rf_pred))

Training Logistic Regression (Clinical Risk Score)...
Training Random Forest (Multidisciplinary Team)...
Training complete! 

--- Diagnostic Accuracy ---
Logistic Regression Accuracy: 81.67%
Random Forest Accuracy: 83.33%

--- Detailed Clinical Report (Random Forest) ---
              precision    recall  f1-score   support

           0       0.84      0.93      0.88        41
           1       0.80      0.63      0.71        19

    accuracy                           0.83        60
   macro avg       0.82      0.78      0.79        60
weighted avg       0.83      0.83      0.83        60

